# 04 — Train Quantile Regressors / เทรน Quantile Regressor (LightGBM)

**EN.** Trains q05/q50/q95 LightGBM quantile regressors for every `(station, horizon)` in the v3 layout using the `lightgbm_quantile` backend (`app/ml/forecast/backends/lgbm_backend.py::LGBMForecaster`). Hyperparameters and quantile levels come from `configs/train/quantile.yaml`. After training, the notebook renders coverage and pinball-loss diagnostics inline (per-(station, horizon) table, per-station coverage bar chart vs. target band, daytime/nighttime split, high-HI ≥40°C split, PI-width distribution).

**TH.** เทรน LightGBM quantile regressor ทั้ง q05/q50/q95 ต่อ `(สถานี, horizon)` ในเลย์เอาต์ v3 ผ่าน backend ชื่อ `lightgbm_quantile` (อยู่ที่ `app/ml/forecast/backends/lgbm_backend.py::LGBMForecaster`). ค่า hyperparameter และระดับ quantile โหลดจาก `configs/train/quantile.yaml`. หลังเทรนเสร็จ notebook จะแสดง coverage / pinball loss / กราฟ diagnostics ครบในไฟล์เดียว (ตารางต่อ `(station, horizon)`, แท่งกราฟ coverage ต่อสถานีเทียบกับ band เป้าหมาย, split กลางวัน/กลางคืน, split HI สูง ≥40°C, การกระจายของ PI width).

**Prerequisites / ข้อกำหนดก่อนรัน**

- `00_setup.ipynb` ran (Drive mounted, repo cloned, requirements installed).
- `01_ingest.ipynb` ran (parquet store populated).
- `03_train_baseline.ipynb` ran first — feature engineering parameters (`_DEFAULT_LAGS_H`, `_DEFAULT_ROLLING_H` in `app/ml/forecast/features.py`) must match those used by the baseline so that v3 quantile artifacts share the same column order.
- `app/models/forecast_v3/` symlinked to Drive (so artifacts persist across Colab sessions).

**What gets written / สิ่งที่ถูกบันทึก**

- `app/models/forecast_v3/{station_id}/h{H}/temp_c_q{05,50,95,97}_s{seed}.txt` (LightGBM booster files, written by `LGBMForecaster.save`)
- `app/models/forecast_v3/{station_id}/h{H}/rh_q{05,50,95,97}_s{seed}.txt`
- `app/models/forecast_v3/{station_id}/h{H}/calibrator.json`, `danger_gate/`, `bundle.json`
- `app/models/forecast_v3/{station_id}/h{H}/registry.json` (registry sidecar)
- `app/models/forecast_v3/choice_matrix.json` — sets `(station, horizon) → "lightgbm_quantile"`

**Coverage targets / เกณฑ์ coverage**

- 90 % nominal PI → empirical coverage in **[0.85, 0.93]** (5 pp tolerance, see `quantile.yaml::diagnostics`).
- Median PI width should stay below **4 °C** to keep `low_confidence` flags off (see `app/ml/forecast/predict.py`).

In [ ]:
# --- bootstrap (re-run if you opened this notebook before 00_setup) -------
import os, sys
REPO_DIR = "/content/Heat-wave-backend"
if not os.path.exists(REPO_DIR):
    !bash {REPO_DIR}/scripts/colab_bootstrap.sh || true
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
os.chdir(REPO_DIR)
print(os.getcwd())

## 1. Config + manifest / โหลด config และเตรียม manifest

**EN.** Reads `configs/train/quantile.yaml` (single source of truth — no hyperparameter is hardcoded in this notebook). Builds a manifest with one row per `(station, horizon, alpha)` triple — three rows per `(station, horizon)` — so we can show per-quantile pinball loss and skip already-trained slots when re-running.

**TH.** อ่าน `configs/train/quantile.yaml` (เป็นแหล่งความจริงเดียว — ไม่มี hyperparameter ฝังในโค้ด notebook). สร้าง manifest หนึ่งแถวต่อ `(station, horizon, alpha)` — สามแถวต่อ `(station, horizon)` — เพื่อให้รายงาน pinball loss แยกตาม quantile และข้าม slot ที่เทรนไปแล้วเมื่อรันซ้ำ.

In [ ]:
# --- load config + build run id + manifest --------------------------------
import json, datetime as _dt, pathlib
import yaml

CONFIG_PATH = pathlib.Path("configs/train/quantile.yaml")
CFG = yaml.safe_load(CONFIG_PATH.read_text(encoding="utf-8"))

STATIONS_LIST = list(CFG["stations"])
HORIZONS = list(CFG["horizons"])
QUANTILES = list(CFG["quantiles"])  # e.g. [0.05, 0.50, 0.95]
BACKEND_NAME = CFG["backend"]
DATA_START, DATA_END = CFG["data_window"]
DIAGNOSTICS_CFG = CFG["diagnostics"]
TRIALS = int(CFG["optuna"].get("trials", 25))
SEED = int(CFG.get("seed", 42))

RUN_DATE = _dt.date.today().strftime("%Y%m%d")
RUN_ID = f"{RUN_DATE}_quantile_lgbm"

print(f"backend       = {BACKEND_NAME}")
print(f"run_id        = {RUN_ID}")
print(f"stations      = {STATIONS_LIST}")
print(f"horizons      = {HORIZONS}")
print(f"quantiles     = {QUANTILES}")
print(f"data window   = {DATA_START} -> {DATA_END}")
print(f"optuna trials = {TRIALS}")
print(f"seed          = {SEED}")
assert BACKEND_NAME == "lightgbm_quantile", (
    f"This notebook expects backend=lightgbm_quantile, got {BACKEND_NAME!r}."
)

In [ ]:
# --- manifest of (station, horizon, alpha) triples + skip-already-trained -
import json, pathlib

V3_ROOT = pathlib.Path("app/models/forecast_v3")
MANIFEST_PATH = V3_ROOT / f"_manifest_{RUN_ID}.json"
V3_ROOT.mkdir(parents=True, exist_ok=True)

def _slot_dir(station: str, horizon: int) -> pathlib.Path:
    return V3_ROOT / station / f"h{horizon}"

def _slot_already_trained(station: str, horizon: int) -> bool:
    """A slot counts as 'done' when bundle.json + the q05/q50/q95 boosters exist."""
    sd = _slot_dir(station, horizon)
    if not (sd / "bundle.json").exists():
        return False
    # The lightgbm_quantile backend writes per-target × per-quantile × per-seed files.
    # We check the median seed (s42) for q05/q50/q95 on temp_c as a cheap proxy.
    for q in (5, 50, 95):
        if not (sd / f"temp_c_q{q:02d}_s42.txt").exists():
            return False
    return True

if MANIFEST_PATH.exists():
    manifest = json.loads(MANIFEST_PATH.read_text(encoding="utf-8"))
    print(f"resumed manifest with {len(manifest['entries'])} entries")
else:
    entries = []
    for s in STATIONS_LIST:
        for h in HORIZONS:
            for a in QUANTILES:
                entries.append({
                    "station": s,
                    "horizon_h": int(h),
                    "alpha": float(a),
                    "status": "pending",
                    "pinball_loss": None,
                    "empirical_coverage": None,
                    "trained_at_utc": None,
                })
    manifest = {
        "run_id": RUN_ID,
        "backend": BACKEND_NAME,
        "config": str(CONFIG_PATH),
        "entries": entries,
    }
    MANIFEST_PATH.write_text(json.dumps(manifest, indent=2), encoding="utf-8")
    print(f"wrote manifest: {MANIFEST_PATH} ({len(entries)} entries = {len(STATIONS_LIST)} × {len(HORIZONS)} × {len(QUANTILES)})")

# Mark already-trained (station, horizon) groups as 'skipped' across all alphas.
skipped = 0
for e in manifest["entries"]:
    if _slot_already_trained(e["station"], e["horizon_h"]) and e["status"] == "pending":
        e["status"] = "skipped"
        skipped += 1
MANIFEST_PATH.write_text(json.dumps(manifest, indent=2), encoding="utf-8")
print(f"skipped (already trained): {skipped}")

## 2. Data load + features / โหลดข้อมูลและสร้างฟีเจอร์

**EN.** Uses the same parquet loader and feature builder that `03_train_baseline.ipynb` uses. The feature engineering invariant (`ts_utc <= row.ts_utc` only — see `app/ml/forecast/features.py`) is identical, so the column order produced here matches the baseline; this notebook does **not** re-define lags/rolling windows.

**TH.** ใช้ parquet loader และ feature builder ตัวเดียวกับ `03_train_baseline.ipynb`. ข้อกำหนด no-leakage (`ts_utc <= row.ts_utc`) ใน `app/ml/forecast/features.py` ยังคงเดิม ดังนั้นลำดับคอลัมน์ที่ได้จะตรงกับ baseline notebook นี้ **ไม่** กำหนด lags/rolling เอง.

In [ ]:
# --- load observations + build features per station -----------------------
import datetime as _dt
import pandas as pd
from app.data.loaders import read_observations
from app.ml.forecast.features import build_X_once, build_y_for_horizon

def _to_date(d):
    if isinstance(d, _dt.date):
        return d
    return _dt.date.fromisoformat(str(d))

DATA_START_D = _to_date(DATA_START)
DATA_END_D = _to_date(DATA_END)

def load_station_xy(station: str):
    """Return (X_full, df_aug) for a station — y is built per-horizon downstream."""
    obs = read_observations(station, DATA_START_D, DATA_END_D)
    if obs.empty:
        raise RuntimeError(f"no observations for {station} in {DATA_START_D}..{DATA_END_D}")
    obs = obs.sort_values("ts_utc").reset_index(drop=True)
    obs["station_id"] = station
    X_full, df_aug = build_X_once(obs)
    return X_full, df_aug

_DATA_CACHE = {}
for sid in STATIONS_LIST:
    try:
        X_full, df_aug = load_station_xy(sid)
        _DATA_CACHE[sid] = (X_full, df_aug)
        print(f"  {sid}: X_full rows={len(X_full):>6d}  cols={X_full.shape[1]}  ts=[{df_aug['ts_utc'].min()}, {df_aug['ts_utc'].max()}]")
    except Exception as exc:
        print(f"  {sid}: ERROR {exc}")

## 3. Train loop / ลูปเทรน

**EN.** For each `(station, horizon)`:

1. Build the per-horizon target (`y[temp_c, rh]`) using `build_y_for_horizon`.
2. Time-series split with `gap = horizon_h` is handled inside `LGBMForecaster.fit` (see `app/ml/forecast/splitting.py::split_xy`), preserving the no-leakage invariant.
3. Fit the `lightgbm_quantile` backend — it trains q05/q50/q95 (plus q97 used by the danger-gate pathway) on temp_c **and** rh with `objective="quantile"`, `alpha=α` per booster, multi-seed averaged.
4. Compute per-quantile pinball loss on the validation slice for the manifest's `(station, horizon, alpha)` rows.
5. Save via `save_model_v3` so backend bundle and `choice_matrix.json` are updated together (no hand-written JSON).

**TH.** สำหรับทุก `(สถานี, horizon)`:

1. สร้าง target ต่อ horizon (`y[temp_c, rh]`) ด้วย `build_y_for_horizon`.
2. การ split แบบ time-series ที่มี `gap = horizon_h` ถูกจัดการในตัว `LGBMForecaster.fit` (ดู `app/ml/forecast/splitting.py::split_xy`) — รักษา invariant ห้ามรั่วข้อมูลอนาคต.
3. Fit backend `lightgbm_quantile` — เทรน q05/q50/q95 (รวม q97 ที่ใช้กับ danger-gate) ทั้ง temp_c และ rh ด้วย `objective="quantile"`, `alpha=α` ต่อ booster, เฉลี่ยหลาย seed.
4. คำนวณ pinball loss ต่อ quantile บน validation slice แล้วบันทึกเป็นแถว `(station, horizon, alpha)` ใน manifest.
5. บันทึกผ่าน `save_model_v3` ให้ bundle ของ backend และ `choice_matrix.json` อัปเดตพร้อมกัน (ไม่เขียน JSON ด้วยมือ).

In [ ]:
# --- helpers: pinball loss, val-set quantile predictions, slot training ----
import time, datetime as _dt
import numpy as np
import pandas as pd

from app.ml.forecast.features import build_y_for_horizon
from app.ml.forecast.splitting import split_xy, fit_feature_medians, apply_feature_medians
from app.ml.forecast.backends.lgbm_backend import LGBMForecaster, _compute_hi_array
from app.ml.registry import save_model_v3

def pinball_loss(y_true: np.ndarray, y_pred: np.ndarray, alpha: float) -> float:
    """Pinball / quantile loss at level alpha. Lower is better."""
    diff = y_true - y_pred
    return float(np.mean(np.maximum(alpha * diff, (alpha - 1.0) * diff)))

def _val_quantile_preds_hi(forecaster: LGBMForecaster, X_val: pd.DataFrame) -> dict:
    """Return composed-HI predictions at q05/q50/q95 for the val slice.

    The lightgbm_quantile backend trains quantile boosters on (temp_c, rh) and
    composes HI via the Rothfusz formula. We reuse the same composition here
    so that pinball loss / coverage is computed in HI space, matching the way
    `predict_with_pi` reports intervals.
    """
    preds = forecaster._predict_th(X_val)  # uses _align internally
    hi_q05 = _compute_hi_array(preds["temp_c_q05"], preds["rh_q50"])
    hi_q50 = _compute_hi_array(preds["temp_c_q50"], preds["rh_q50"])
    hi_q95 = _compute_hi_array(preds["temp_c_q95"], preds["rh_q50"])
    return {0.05: hi_q05, 0.50: hi_q50, 0.95: hi_q95}

def _val_slice(X_full: pd.DataFrame, df_aug: pd.DataFrame, horizon_h: int):
    """Reproduce the chronological val slice that fit() consumed, so we can
    evaluate per-quantile diagnostics on the same rows the model trained on.
    Returns (X_val, y_val_df, ts_val)."""
    y = build_y_for_horizon(df_aug, horizon_h=horizon_h, targets=("temp_c", "rh"))
    Xy = X_full.join(y, how="inner").dropna(subset=["temp_c", "rh"])
    X = Xy[[c for c in Xy.columns if c not in ("temp_c", "rh")]]
    yy = Xy[["temp_c", "rh"]]
    split = split_xy(X, yy, horizon_h=horizon_h)
    medians = fit_feature_medians(split.X_train)
    X_val = apply_feature_medians(split.X_val, medians)
    ts_val = df_aug.loc[split.X_val.index, "ts_utc"]
    return X_val, split.y_val, ts_val

In [ ]:
# --- main training loop ---------------------------------------------------
import json, time
import numpy as np

# Per-(station, horizon) diagnostics collected for the report below.
DIAG_ROWS = []
VAL_PREDS = {}  # (station, horizon) -> {"y_hi": np.ndarray, "q05": ..., "q50": ..., "q95": ..., "ts": pd.Series}

for sid in STATIONS_LIST:
    if sid not in _DATA_CACHE:
        print(f"  {sid}: skipped (no data)")
        continue
    X_full, df_aug = _DATA_CACHE[sid]
    for h in HORIZONS:
        slot_entries = [e for e in manifest["entries"]
                        if e["station"] == sid and e["horizon_h"] == h]
        if all(e["status"] == "skipped" for e in slot_entries):
            print(f"  {sid} h{h}: SKIP (already trained)")
            continue

        try:
            t0 = time.perf_counter()
            y = build_y_for_horizon(df_aug, horizon_h=h, targets=("temp_c", "rh"))
            Xy = X_full.join(y, how="inner").dropna(subset=["temp_c", "rh"])
            X = Xy[[c for c in Xy.columns if c not in ("temp_c", "rh")]]
            yy = Xy[["temp_c", "rh"]]

            forecaster = LGBMForecaster(n_trials=TRIALS, random_state=SEED)
            forecaster.fit(X, yy, station_id=sid, horizon_h=h)

            # Save via the registry — backend writes its own bundle.json,
            # registry.json is the sidecar, choice_matrix.json is updated.
            save_model_v3(
                forecaster,
                metadata={
                    "run_id": RUN_ID,
                    "config_path": str(CONFIG_PATH),
                    "data_window": [str(DATA_START_D), str(DATA_END_D)],
                    "quantiles": QUANTILES,
                    "trained_at_utc": _dt.datetime.now(_dt.timezone.utc).isoformat(),
                },
                station_id=sid,
                horizon_h=h,
            )

            # Per-quantile diagnostics on the val slice
            X_val, y_val_df, ts_val = _val_slice(X_full, df_aug, horizon_h=h)
            qpreds = _val_quantile_preds_hi(forecaster, X_val)
            y_hi_val = _compute_hi_array(y_val_df["temp_c"].values, y_val_df["rh"].values)

            for alpha in QUANTILES:
                p = qpreds[alpha]
                pl = pinball_loss(y_hi_val, p, alpha)
                # Empirical coverage at this alpha makes sense only as a tail check,
                # but we still record it so the manifest shows the same shape across
                # alphas (full 90% PI coverage is reported in the diagnostics table).
                if alpha == 0.05:
                    cov = float((y_hi_val >= p).mean())
                elif alpha == 0.95:
                    cov = float((y_hi_val <= p).mean())
                else:
                    cov = float((np.abs(y_hi_val - p) <= 1.0).mean())  # ±1°C around median
                for e in slot_entries:
                    if abs(e["alpha"] - alpha) < 1e-9:
                        e["status"] = "trained"
                        e["pinball_loss"] = round(pl, 4)
                        e["empirical_coverage"] = round(cov, 4)
                        e["trained_at_utc"] = _dt.datetime.now(_dt.timezone.utc).isoformat()

            # 90 % PI diagnostics (the headline metric)
            in_band = (y_hi_val >= qpreds[0.05]) & (y_hi_val <= qpreds[0.95])
            cov90 = float(in_band.mean())
            mean_pi = float(np.mean(qpreds[0.95] - qpreds[0.05]))
            median_pi = float(np.median(qpreds[0.95] - qpreds[0.05]))
            DIAG_ROWS.append({
                "station": sid,
                "horizon_h": h,
                "n_val": int(len(y_hi_val)),
                "pinball_q05": pinball_loss(y_hi_val, qpreds[0.05], 0.05),
                "pinball_q50": pinball_loss(y_hi_val, qpreds[0.50], 0.50),
                "pinball_q95": pinball_loss(y_hi_val, qpreds[0.95], 0.95),
                "coverage_90": cov90,
                "mean_pi_width_c": mean_pi,
                "median_pi_width_c": median_pi,
                "train_seconds": round(time.perf_counter() - t0, 1),
            })
            VAL_PREDS[(sid, h)] = {
                "y_hi": y_hi_val,
                "q05": qpreds[0.05],
                "q50": qpreds[0.50],
                "q95": qpreds[0.95],
                "ts": ts_val.reset_index(drop=True),
            }

            MANIFEST_PATH.write_text(json.dumps(manifest, indent=2), encoding="utf-8")
            print(f"  {sid} h{h}: cov90={cov90:.3f}  mean_PI={mean_pi:.2f}°C  pinball(q50)={DIAG_ROWS[-1]['pinball_q50']:.3f}  ({DIAG_ROWS[-1]['train_seconds']:.1f}s)")
        except Exception as exc:
            for e in slot_entries:
                if e["status"] == "pending":
                    e["status"] = "failed"
                    e["error"] = str(exc)
            MANIFEST_PATH.write_text(json.dumps(manifest, indent=2), encoding="utf-8")
            print(f"  {sid} h{h}: ERROR {exc!r}")

print("\nDone. Manifest:", MANIFEST_PATH)

## 4. Coverage diagnostics / รายงาน coverage และ PI width

**EN.** Empirical 90 % PI coverage = fraction of `y_val` that falls in `[q05, q95]`. Target band is `[0.85, 0.93]` (5 pp tolerance around the nominal 0.90, see `quantile.yaml::diagnostics`). Charts below render inline so the notebook itself acts as the eval report (per the project memory: full metrics + all charts must show after every train).

**TH.** Coverage จริงของ 90 % PI = สัดส่วน `y_val` ที่ตกใน `[q05, q95]`. ช่วงเป้าหมายคือ `[0.85, 0.93]` (เผื่อความคลาด 5 pp รอบ 0.90 ตาม `quantile.yaml::diagnostics`). กราฟทุกตัวอยู่ในหน้านี้เลย เพื่อให้ notebook นี้ทำหน้าที่เป็น eval report ตามข้อตกลงของโปรเจกต์ (หลังเทรนต้องเห็นเมตริกครบ + กราฟครบในไฟล์เดียว).

In [ ]:
# --- per-(station, horizon) diagnostics table -----------------------------
import pandas as pd

DIAG_DF = pd.DataFrame(DIAG_ROWS).sort_values(["station", "horizon_h"]).reset_index(drop=True)
TARGET_LO = 0.90 - DIAGNOSTICS_CFG["coverage_tolerance"]  # 0.85
TARGET_HI = 0.90 + DIAGNOSTICS_CFG["coverage_tolerance"]  # 0.95 nominal — clamp display to 0.93 per spec
TARGET_HI_DISPLAY = 0.93
PI_WIDTH_MAX = float(DIAGNOSTICS_CFG["pi_width_max_c"])

def _flag(c):
    if pd.isna(c):
        return "--"
    if c < TARGET_LO:
        return "UNDER"
    if c > TARGET_HI_DISPLAY:
        return "OVER"
    return "OK"

DIAG_DF["flag"] = DIAG_DF["coverage_90"].apply(_flag)
DIAG_DF_DISPLAY = DIAG_DF[[
    "station", "horizon_h", "n_val",
    "pinball_q05", "pinball_q50", "pinball_q95",
    "coverage_90", "mean_pi_width_c", "median_pi_width_c", "flag",
]]
DIAG_DF_DISPLAY

In [ ]:
# --- per-station coverage bar chart vs target band [0.85, 0.93] -----------
import matplotlib.pyplot as plt
import numpy as np

if not DIAG_DF.empty:
    pivot = DIAG_DF.pivot(index="station", columns="horizon_h", values="coverage_90")
    fig, ax = plt.subplots(figsize=(9, 4.5))
    pivot.plot(kind="bar", ax=ax, edgecolor="black", width=0.85)
    ax.axhspan(TARGET_LO, TARGET_HI_DISPLAY, color="green", alpha=0.12, label=f"target band [{TARGET_LO:.2f}, {TARGET_HI_DISPLAY:.2f}]")
    ax.axhline(0.90, color="green", linestyle="--", linewidth=1, label="nominal 0.90")
    ax.set_ylabel("Empirical 90% PI coverage")
    ax.set_xlabel("Station")
    ax.set_ylim(0.5, 1.02)
    ax.set_title("90% PI coverage per station × horizon vs target band")
    ax.legend(loc="lower right", fontsize=8, ncol=2)
    ax.tick_params(axis="x", rotation=0)
    fig.tight_layout()
    plt.show()
else:
    print("DIAG_DF is empty — nothing trained this run.")

In [ ]:
# --- daytime (10:00-18:00 local) vs nighttime coverage split --------------
# Local hour = (UTC + 7) % 24 — Asia/Bangkok offset matches features.py.
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

split_rows = []
for (sid, h), pred in VAL_PREDS.items():
    ts = pd.to_datetime(pred["ts"], utc=True)
    local_hour = (ts.dt.hour + 7) % 24
    day_mask = local_hour.between(10, 18, inclusive="left").to_numpy()
    in_band = (pred["y_hi"] >= pred["q05"]) & (pred["y_hi"] <= pred["q95"])
    if day_mask.sum() > 0:
        split_rows.append({"station": sid, "horizon_h": h, "window": "day (10-18 local)",
                           "coverage_90": float(in_band[day_mask].mean()), "n": int(day_mask.sum())})
    if (~day_mask).sum() > 0:
        split_rows.append({"station": sid, "horizon_h": h, "window": "night (18-10 local)",
                           "coverage_90": float(in_band[~day_mask].mean()), "n": int((~day_mask).sum())})

SPLIT_DF = pd.DataFrame(split_rows)
SPLIT_DF.head(20)

In [ ]:
# --- chart: day vs night coverage by station ------------------------------
import matplotlib.pyplot as plt

if not SPLIT_DF.empty:
    agg = SPLIT_DF.groupby(["station", "window"])["coverage_90"].mean().unstack("window")
    fig, ax = plt.subplots(figsize=(9, 4.5))
    agg.plot(kind="bar", ax=ax, edgecolor="black", width=0.8)
    ax.axhspan(TARGET_LO, TARGET_HI_DISPLAY, color="green", alpha=0.12, label=f"target [{TARGET_LO:.2f}, {TARGET_HI_DISPLAY:.2f}]")
    ax.axhline(0.90, color="green", linestyle="--", linewidth=1)
    ax.set_ylabel("Mean 90% PI coverage")
    ax.set_xlabel("Station")
    ax.set_ylim(0.5, 1.02)
    ax.set_title("Day vs night coverage (averaged across horizons)")
    ax.tick_params(axis="x", rotation=0)
    ax.legend(loc="lower right", fontsize=8)
    fig.tight_layout()
    plt.show()
else:
    print("SPLIT_DF empty — no validation rows to split.")

In [ ]:
# --- coverage by hour-of-day (across all stations × horizons) -------------
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

hour_rows = []
for (sid, h), pred in VAL_PREDS.items():
    ts = pd.to_datetime(pred["ts"], utc=True)
    local_hour = ((ts.dt.hour + 7) % 24).to_numpy()
    in_band = (pred["y_hi"] >= pred["q05"]) & (pred["y_hi"] <= pred["q95"])
    for hh in range(24):
        m = local_hour == hh
        if m.sum() > 0:
            hour_rows.append({"station": sid, "horizon_h": h, "local_hour": hh,
                              "coverage": float(in_band[m].mean()), "n": int(m.sum())})
HOUR_DF = pd.DataFrame(hour_rows)

if not HOUR_DF.empty:
    avg_by_hour = HOUR_DF.groupby("local_hour")["coverage"].mean()
    fig, ax = plt.subplots(figsize=(9, 3.8))
    avg_by_hour.plot(kind="bar", ax=ax, color="#3a7bd5", edgecolor="black")
    ax.axhspan(TARGET_LO, TARGET_HI_DISPLAY, color="green", alpha=0.12)
    ax.axhline(0.90, color="green", linestyle="--", linewidth=1, label="nominal 0.90")
    ax.set_xlabel("Local hour (Asia/Bangkok)")
    ax.set_ylabel("Mean 90% PI coverage")
    ax.set_ylim(0.5, 1.02)
    ax.set_title("Coverage by hour of day")
    ax.legend(loc="lower right", fontsize=8)
    fig.tight_layout()
    plt.show()
else:
    print("HOUR_DF empty.")

In [ ]:
# --- high-HI (>=40°C) vs normal coverage split + HI-band chart ------------
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

hi_rows = []
BANDS = [(-np.inf, 30), (30, 35), (35, 40), (40, np.inf)]
BAND_LABELS = ["<30", "30-35", "35-40", ">=40"]
for (sid, h), pred in VAL_PREDS.items():
    y = pred["y_hi"]
    in_band_pi = (y >= pred["q05"]) & (y <= pred["q95"])
    # high-HI vs rest (the spec calls this out)
    hot = y >= 40
    if hot.sum() > 0:
        hi_rows.append({"station": sid, "horizon_h": h, "regime": "HI>=40°C",
                        "coverage_90": float(in_band_pi[hot].mean()), "n": int(hot.sum())})
    if (~hot).sum() > 0:
        hi_rows.append({"station": sid, "horizon_h": h, "regime": "HI<40°C",
                        "coverage_90": float(in_band_pi[~hot].mean()), "n": int((~hot).sum())})
    # Finer bands for the chart
    for (lo, up), lab in zip(BANDS, BAND_LABELS):
        m = (y >= lo) & (y < up)
        if m.sum() > 0:
            hi_rows.append({"station": sid, "horizon_h": h, "regime": f"band:{lab}",
                            "coverage_90": float(in_band_pi[m].mean()), "n": int(m.sum())})

HI_DF = pd.DataFrame(hi_rows)
HOT_SUMMARY = HI_DF[HI_DF["regime"].isin(["HI>=40°C", "HI<40°C"])]
HOT_SUMMARY

In [ ]:
# --- chart: coverage by HI band (>= 40°C is the alert regime) -------------
import matplotlib.pyplot as plt

BAND_DF = HI_DF[HI_DF["regime"].str.startswith("band:")].copy()
BAND_DF["band"] = BAND_DF["regime"].str.replace("band:", "", regex=False)
if not BAND_DF.empty:
    agg = (BAND_DF.groupby("band")
                  .apply(lambda g: np.average(g["coverage_90"], weights=g["n"]))
                  .reindex(BAND_LABELS))
    fig, ax = plt.subplots(figsize=(7, 4))
    agg.plot(kind="bar", ax=ax, color=["#4e79a7", "#f28e2b", "#e15759", "#b07aa1"], edgecolor="black")
    ax.axhspan(TARGET_LO, TARGET_HI_DISPLAY, color="green", alpha=0.12, label=f"target [{TARGET_LO:.2f}, {TARGET_HI_DISPLAY:.2f}]")
    ax.axhline(0.90, color="green", linestyle="--", linewidth=1)
    ax.set_xlabel("HI band (°C)")
    ax.set_ylabel("Weighted mean 90% PI coverage")
    ax.set_ylim(0.5, 1.02)
    ax.set_title("Coverage by heat-index band — watch the >=40°C bar")
    ax.tick_params(axis="x", rotation=0)
    ax.legend(loc="lower right", fontsize=8)
    fig.tight_layout()
    plt.show()
else:
    print("BAND_DF empty.")

In [ ]:
# --- PI width distribution + low_confidence threshold -------------------
import numpy as np
import matplotlib.pyplot as plt

all_widths = []
for (sid, h), pred in VAL_PREDS.items():
    all_widths.append(pred["q95"] - pred["q05"])
if all_widths:
    widths = np.concatenate(all_widths)
    fig, ax = plt.subplots(figsize=(8, 3.8))
    ax.hist(widths, bins=40, color="#3a7bd5", edgecolor="black", alpha=0.85)
    ax.axvline(PI_WIDTH_MAX, color="red", linestyle="--", linewidth=1.2,
               label=f"low_confidence threshold = {PI_WIDTH_MAX:.1f}°C")
    ax.axvline(np.median(widths), color="black", linestyle=":", linewidth=1.2,
               label=f"median = {np.median(widths):.2f}°C")
    ax.set_xlabel("PI width (q95 − q05) in °C")
    ax.set_ylabel("Count of validation rows")
    ax.set_title("Distribution of 90% PI widths across all (station, horizon) val slices")
    ax.legend(loc="upper right", fontsize=8)
    fig.tight_layout()
    plt.show()
    print(f"  median width = {np.median(widths):.2f}°C")
    print(f"  mean width   = {np.mean(widths):.2f}°C")
    print(f"  P95 width    = {np.percentile(widths, 95):.2f}°C")
    over = float((widths > PI_WIDTH_MAX).mean())
    print(f"  fraction over {PI_WIDTH_MAX:.1f}°C threshold (low_confidence): {over:.1%}")
else:
    print("No PI widths captured this run.")

In [ ]:
# --- pinball loss heatmap (station × horizon × quantile) ------------------
import matplotlib.pyplot as plt
import numpy as np

if not DIAG_DF.empty:
    fig, axes = plt.subplots(1, 3, figsize=(13, 3.6), sharey=True)
    for ax, col, title in zip(
        axes,
        ["pinball_q05", "pinball_q50", "pinball_q95"],
        ["Pinball q05", "Pinball q50 (median)", "Pinball q95"],
    ):
        m = DIAG_DF.pivot(index="station", columns="horizon_h", values=col)
        im = ax.imshow(m.values, aspect="auto", cmap="viridis_r")
        ax.set_xticks(range(len(m.columns)))
        ax.set_xticklabels([f"h{c}" for c in m.columns])
        ax.set_yticks(range(len(m.index)))
        ax.set_yticklabels(m.index)
        ax.set_title(title)
        for i in range(len(m.index)):
            for j in range(len(m.columns)):
                v = m.values[i, j]
                if not np.isnan(v):
                    ax.text(j, i, f"{v:.2f}", ha="center", va="center",
                            color="white" if v > np.nanmean(m.values) else "black", fontsize=8)
        fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    fig.suptitle("Pinball loss (lower is better)", y=1.02)
    fig.tight_layout()
    plt.show()
else:
    print("DIAG_DF empty — no pinball heatmap.")

## 5. Mini-report / สรุปสั้น

**EN.**

- Trained `lightgbm_quantile` for every `(station, horizon)` listed in `quantile.yaml`. q05/q50/q95 (and q97 used downstream by the danger gate) come from `LGBMRegressor(objective="quantile", alpha=α)` with shared best params from the Optuna search at q50, multi-seed averaged.
- Diagnostics rendered above: per-(station, horizon) pinball + coverage table, per-station coverage bars vs. target band `[0.85, 0.93]`, day vs night split, by-hour coverage, by-HI-band coverage, and PI-width histogram. The same `low_confidence` thresholds used by `app/ml/forecast/predict.py` (60-min stale-obs cutoff, 4°C PI cap, 1.6× v3 median PI width) are visualised on the histogram.
- Artifacts persist under `app/models/forecast_v3/{station}/h{H}/` and `choice_matrix.json` is updated to point each `(station, horizon)` slot at `lightgbm_quantile`. The v3-aware predict path picks them up automatically.

**TH.**

- เทรน backend `lightgbm_quantile` ครบทุก `(สถานี, horizon)` ใน `quantile.yaml`. q05/q50/q95 (และ q97 ที่ danger gate ใช้ภายใน) มาจาก `LGBMRegressor(objective="quantile", alpha=α)` โดยใช้ best params ร่วมจาก Optuna ที่ q50 และเฉลี่ยหลาย seed
- กราฟ diagnostics ครบในไฟล์เดียว — ตาราง pinball + coverage ต่อ `(station, horizon)`, แท่ง coverage ต่อสถานีเทียบ band `[0.85, 0.93]`, split กลางวัน/กลางคืน, coverage ต่อชั่วโมง, coverage ต่อ HI band, และ histogram ของ PI width. Threshold ของ `low_confidence` ที่ `app/ml/forecast/predict.py` ใช้ (obs เก่าเกิน 60 นาที, PI กว้างเกิน 4°C, หรือ 1.6× ของ median v3 PI) ถูก plot ไว้บน histogram
- artifact ถูกเก็บที่ `app/models/forecast_v3/{station}/h{H}/` และ `choice_matrix.json` ชี้ไปที่ `lightgbm_quantile` แล้ว — predict path ที่รองรับ v3 จะหยิบไปใช้ทันที

**Next / ขั้นถัดไป.** เปิด `06_calibrate.ipynb` เพื่อปรับ EnbPI / Mondrian CQR calibrator ของ backend นี้บนข้อมูลใหม่ และตรวจ coverage หลัง calibration อีกครั้ง